# Pipeline de Ejecución Integrado - ALDIMI Pharma Demand

Este notebook automatiza el flujo de trabajo completo:
1. **Clasificación de Riesgo:** Evalúa a los pacientes nuevos usando el **Modelo 2 (Árbol de Decisión)**.
2. **Generación de Censo:** Calcula el estado de vulnerabilidad actual de la población.
3. **Pronóstico de Demanda:** Usa el censo y el historial de ventas para predecir necesidades de stock con el **Modelo 1 (Regresión Lineal)**.

In [9]:
import pandas as pd
import numpy as np
import joblib
import os
from pathlib import Path
from datetime import datetime

# Configuración de rutas
MODELS_DIR = Path("../modelos")
DATA_INPUT = Path("../datos/datos_modelo2/aldimi_pacientes_censo_INPUT_M1.csv")
DATA_HISTORIC = Path("../datos/datos_modelo1/processed/aldimi_demanda_dataset_clean.csv")

print("✅ Entorno de ejecución configurado.")

✅ Entorno de ejecución configurado.


## Paso 1: Clasificación de Pacientes (Modelo 2)
Cargamos los datos de los pacientes que acaban de llegar y les asignamos un nivel de riesgo automático.

In [23]:
# 1. Cargar el Modelo de Riesgo
modelo_riesgo = joblib.load(MODELS_DIR / "modelo_riesgo_pacientes.joblib")

# 2. Leer pacientes nuevos (simulación)
pacientes_nuevos = pd.read_csv(DATA_INPUT)

# 3. Predecir Niveles de Riesgo
predicciones = modelo_riesgo.predict(pacientes_nuevos)
pacientes_nuevos['Level_Predicho'] = predicciones

print(f"--- Clasificación Completada ---")
print(pacientes_nuevos['Level_Predicho'].value_counts())
pacientes_nuevos[['Age', 'Snoring', 'Obesity', 'Air Pollution', 'Wheezing', 'Coughing of Blood','Level_Predicho']].tail(15)

--- Clasificación Completada ---
Level_Predicho
Low       57
Medium    23
High      20
Name: count, dtype: int64


,Age,Snoring,Obesity,Air Pollution,Wheezing,Coughing of Blood,Level_Predicho
85,25,3,3,3,4,1,Low
86,55,2,1,3,1,3,Low
87,23,1,4,4,2,4,Low
88,52,3,2,3,3,1,Low
89,35,7,4,2,6,6,High
90,26,1,7,7,7,7,High
91,19,4,3,3,5,3,Medium
92,39,3,7,6,1,9,High
93,38,5,3,1,4,4,Medium
94,33,4,4,2,2,4,Low


## Paso 2: Generación del Censo (Agregación)
Transformamos las predicciones individuales en los indicadores que necesita el modelo de demanda.

In [11]:
counts = pacientes_nuevos['Level_Predicho'].value_counts().to_dict()

n_low = counts.get('Low', 0)
n_medium = counts.get('Medium', 0)
n_high = counts.get('High', 0)

print(f"Censo Actualizado: Low={n_low}, Medium={n_medium}, High={n_high}")

Censo Actualizado: Low=57, Medium=23, High=20


## Paso 3: Pronóstico de Demanda Acumulada (Modelo 1)
Usamos los 6 modelos de regresión re-entrenados para predecir la SUMA de demanda de los próximos 7 y 14 días.


In [12]:
# 1. Obtener datos históricos de demanda para calcular los 'Lags'
df_hist = pd.read_csv(DATA_HISTORIC)

# CAMBIO: Usar 'fecha' en lugar de 'date'
df_hist['fecha'] = pd.to_datetime(df_hist['fecha'])
df_hist = df_hist.sort_values('fecha')

# Tomamos el último día disponible como 'Hoy'
ultima_fecha = df_hist['fecha'].max()
dia_semana = ultima_fecha.dayofweek
es_finde = 1 if dia_semana >= 5 else 0

drugs = ['N02BE', 'N05B', 'M01AB']
horizons = [7, 14]
pronosticos = []

print(f"Generando pronósticos desde la fecha: {ultima_fecha.date()}\n")

# ... (código anterior de carga de df_hist igual)

for drug in drugs:
    # 1. Extraer variables de memoria (Lags)
    demanda_hoy = df_hist[f'{drug}_demand'].iloc[-1]
    promedio_semanal = df_hist[f'{drug}_demand'].tail(7).mean()
    
    # 2. Crear el vector de características (X) con el ORDEN EXACTO del entrenamiento
    # El modelo NO espera la columna 'N02BE_demand', solo sus derivados.
    features = ['n_low', 'n_medium', 'n_high', 'dia_semana', 'es_fin_de_semana', 'demanda_hoy', 'promedio_semanal']
    data_values = [[n_low, n_medium, n_high, dia_semana, es_finde, demanda_hoy, promedio_semanal]]
    
    X_input = pd.DataFrame(data_values, columns=features)
    
    for h in horizons:
        # Cargar modelo específico
        path_modelo = MODELS_DIR / f"modelo_demanda_{drug}_{h}dias_acumulado.joblib"
        if path_modelo.exists():
            model = joblib.load(path_modelo)
            pred = model.predict(X_input)[0]
            
            pronosticos.append({
                'Medicamento': drug, 
                'Periodo': f'Próximos {h} días',
                'Total Unidades a Comprar (Acumulado)': int(np.ceil(max(0, pred)))
            })

df_final = pd.DataFrame(pronosticos)
display(df_final)


Generando pronósticos desde la fecha: 2019-10-08



,Medicamento,Horizonte,Unidades a Comprar
0,N02BE,7 días,246
1,N02BE,14 días,244
2,N05B,7 días,30
3,N05B,14 días,30
4,M01AB,7 días,21
5,M01AB,14 días,21


## Resumen de Recomendación Logística
Los valores mostrados arriba son la cantidad de unidades que ALDIMI debería asegurar para cubrir la demanda en los periodos indicados, basándose en la salud actual de sus pacientes.